# (1) import libraries

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import numpy as np
from scipy.stats import multivariate_normal

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import time
from IPython.display import clear_output

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# (2) data preparation

In [ ]:
# training ------------------------------
A = 32j
x1, x2 = np.mgrid[-3.0:3.0:A, -3.0:3.0:A]
x1x2 = np.column_stack([x1.flat, x2.flat])

mu = np.array([0.0, 0.0])
covariance = np.diag([1, 1])

y = multivariate_normal.pdf(x1x2, mean=mu, cov=covariance)
y = y.reshape(x1.shape)

x_train = x1x2
y_train = y.flatten()

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(x1, x2, y)
plt.title('training data')
plt.show()

# testing -------------------------------
A = 10j

x1, x2 = np.mgrid[-3.0:3.0:A, -3.0:3.0:A]
x1x2 = np.column_stack([x1.flat, x2.flat])

mu = np.array([0.0, 0.0])
covariance = np.diag([1, 1])

y = multivariate_normal.pdf(x1x2, mean=mu, cov=covariance)
y = y.reshape(x1.shape)

x_test = x1x2
y_test = y.flatten()

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(x1, x2, y)
plt.title('testing data')
plt.show()

# (3) MLP

In [ ]:
def keras_init(m):
    # Keras default initialization: glorot_uniform weights, zero biases
    # (PyTorch's default init trains noticeably slower with the same SGD/Adam settings)
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


class GaussMLP(nn.Module):
    def __init__(self, B=16):
        super().__init__()
        # 2 input nodes -> FC layer with B hidden nodes
        self.fc = nn.Linear(2, B)

        # Output
        self.gauss_apxm = nn.Linear(B, 1)

    def forward(self, x):
        x = torch.sigmoid(self.fc(x))
        return torch.sigmoid(self.gauss_apxm(x)).squeeze(-1)


def gauss_mlp(B=16):
    # define model
    model = GaussMLP(B=B).to(device)
    model.apply(keras_init)

    return model


def to_tensor(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)


def fit(model, x, y, loss_fn, optimizer, batch_size, epochs):
    # equivalent of Keras model.fit(..., verbose=0)
    loader = DataLoader(TensorDataset(to_tensor(x), to_tensor(y)),
                        batch_size=batch_size, shuffle=True)
    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()


@torch.no_grad()
def evaluate(model, x, y, loss_fn):
    # equivalent of Keras model.evaluate(..., verbose=0): returns the loss (MSE)
    model.eval()
    return loss_fn(model(to_tensor(x)), to_tensor(y)).item()

# (4) training and evaluating MLP

### (4.1) Number of training samples (A) (no. of hidden nodes (B) = 16 and no. of epochs (C) = 100)

In [ ]:
train_error = []
test_error = []
tot_train = 1001
A_array = []

B = 16
C = 100
for A in np.arange(100, tot_train, 100):
    # define the model
    model = gauss_mlp(B=B)

    # loss function and optimizer
    loss_fn = nn.MSELoss()
    lr = 0.01
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # train the model
    batch_size = 128
    epochs = int(C)

    # items for training
    sel_indx = np.random.choice(range(1, tot_train), size=A, replace=False)
    fit(model, x_train[sel_indx], y_train[sel_indx], loss_fn, optimizer,
        batch_size=batch_size, epochs=epochs)

    train_error.append(evaluate(model, x_train, y_train, loss_fn))
    test_error.append(evaluate(model, x_test, y_test, loss_fn))
    A_array.append(A)

    # plot the relationship between A (training samples) and loss
    clear_output()
    plt.plot(A_array, train_error, label='training')
    plt.plot(A_array, test_error, label='testing')
    plt.legend()
    plt.title('Number of training samples (A)')
    plt.xlabel('#-samples')
    plt.ylabel('Error')
    plt.show()

### (4.2) Number of hidden nodes (B) (no. of training samples (A) = 800 and no. of epochs (C) = 100)

In [ ]:
train_error = []
test_error = []
B_array = []
tot_train = 1001

A = 800
B = 8
C = 100
while B <= 512:
    # define the model
    model = gauss_mlp(B=B)

    # loss function and optimizer
    loss_fn = nn.MSELoss()
    lr = 0.01
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # train the model
    batch_size = 128
    epochs = int(C)

    # items for training
    sel_indx = np.random.choice(range(1, tot_train), size=A, replace=False)
    fit(model, x_train[sel_indx], y_train[sel_indx], loss_fn, optimizer,
        batch_size=batch_size, epochs=epochs)

    train_error.append(evaluate(model, x_train, y_train, loss_fn))
    test_error.append(evaluate(model, x_test, y_test, loss_fn))
    B_array.append(B)

    # plot the relationship between B (hidden nodes) and loss
    clear_output()
    plt.plot(B_array, train_error, label='training')
    plt.plot(B_array, test_error, label='testing')
    plt.legend()
    plt.title('Number of hidden nodes (B)')
    plt.xlabel('Number of hidden nodes')
    plt.ylabel('Error')
    plt.show()

    B *= 2

### (4.3) Number of epochss (C) (no. of training samples (A) = 800 and no. of hidden nodes (B) = 128)

In [ ]:
train_error = []
test_error = []
tot_train = 1001
C_array = []

A = 800
B = 128
for C in np.arange(10, 200, 10):
    # define the model
    model = gauss_mlp(B=B)

    # loss function and optimizer
    loss_fn = nn.MSELoss()
    lr = 0.01
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # train the model
    batch_size = 128
    epochs = int(C)

    # items for training
    sel_indx = np.random.choice(range(1, tot_train), size=A, replace=False)
    fit(model, x_train[sel_indx], y_train[sel_indx], loss_fn, optimizer,
        batch_size=batch_size, epochs=epochs)

    train_error.append(evaluate(model, x_train, y_train, loss_fn))
    test_error.append(evaluate(model, x_test, y_test, loss_fn))
    C_array.append(C)

    # plot the relationship between C (epochs) and loss
    clear_output()
    plt.plot(C_array, train_error, label='training')
    plt.plot(C_array, test_error, label='testing')
    plt.legend()
    plt.title('Number of epochs (C)')
    plt.xlabel('Epochs')
    plt.ylabel('Error')
    plt.show()